code adapted from :
* https://github.com/spmallick/learnopencv/blob/master/Handwritten_Text_Recognition_using_OCR/Fine_Tune_TrOCR_Handwritten.ipynb
* https://github.com/spmallick/learnopencv/blob/master/Handwritten_Text_Recognition_using_OCR/Pretrained_Model_Inference.ipynb
* https://github.com/NielsRogge/Transformers-Tutorials/blob/master/TrOCR/Evaluating_TrOCR_base_handwritten_on_the_IAM_test_set.ipynb
* https://github.com/NielsRogge/Transformers-Tutorials/blob/master/TrOCR/Fine_tune_TrOCR_on_IAM_Handwriting_Database_using_native_PyTorch.ipynb
* https://github.com/NielsRogge/Transformers-Tutorials/blob/master/TrOCR/Fine_tune_TrOCR_on_IAM_Handwriting_Database_using_Seq2SeqTrainer.ipynb

In [ ]:
!pip install --upgrade transformers datasets jiwer evaluate Levenshtein peft huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 83.5 MB/s eta 0:00:00:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.7/161.7 kB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 411.1/411.1 kB 26.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.6/193.6 kB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 79.8 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.6 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.5 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 6.5 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 30.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 13.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/

In [2]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
from transformers import TrOCRProcessor, AutoTokenizer, VisionEncoderDecoderModel
import torch
from torch.utils.data import Dataset
from PIL import Image

from huggingface_hub import login
access_token = user_secrets.get_secret("HF_Token")
login=login(token=access_token)

2025-06-21 09:58:43.846004: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1750499924.005153      35 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1750499924.052980      35 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


In [3]:
from copy import deepcopy

# Loralizer

In [ ]:
from transformers import TrOCRProcessor, VisionEncoderDecoderModel, AutoTokenizer, TrOCRForCausalLM
from peft import LoraConfig, get_peft_model

# For example, using a TrOCR model pre-trained on handwritten text:
processor = TrOCRProcessor.from_pretrained("ds1219/Pandithaya-Printed-Words-128",
                                           size={"height": 128, "width": 128})
# (Optionally, if you want to change the decoder tokenizer—for example, to a RoBERTa tokenizer for a different language—you can load it separately)
tokenizer = AutoTokenizer.from_pretrained("ds1219/Pandithaya-Printed-Words-128")

# If you want to swap in a different tokenizer for the decoder, assign it to the processor:
processor.tokenizer = tokenizer

# Load the pre-trained TrOCR model:
model = VisionEncoderDecoderModel.from_pretrained("ds1219/Pandithaya-Printed-Words-128")

model.config.decoder_start_token_id = processor.tokenizer.cls_token_id
model.config.pad_token_id = processor.tokenizer.pad_token_id
model.config.vocab_size = model.config.decoder.vocab_size

In [35]:
dora_config = LoraConfig(
    r=8,
    lora_alpha=32,
    lora_dropout=0.05,
    use_dora=True,
    target_modules=["query", "value"], # or try ["key", "query", "value"]
)

lora_config = LoraConfig(
    r=8,
    lora_alpha=32,
    lora_dropout=0.1,
    bias="none",
    target_modules=["q_proj", "v_proj"]  # Adjust these names if needed (check your model's module names)
)

In [36]:
model.encoder = get_peft_model(model.encoder, dora_config)
model.decoder = get_peft_model(model.decoder, lora_config)

In [9]:
model.encoder = model.encoder.merge_and_unload()
model.decoder = model.decoder.merge_and_unload()

In [37]:
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable params: {trainable_params} / Total params: {total_params}")

Trainable params: 1075200 / Total params: 316293632


# Load Dataset

In [8]:
import pandas as pd

def import_dataset(path):
  df = pd.read_csv(path, sep='\t', usecols=[0,1])
  #df = pd.read_csv(path, sep='\t', usecols=[0,1])
  df.rename(columns={'file': 'file_name', 'labels': 'text'}, inplace=True)
  #df = df.iloc[1:]
  return(df)

In [9]:
train_df = import_dataset('/kaggle/input/real-synth/real_train.tsv')

#train_df = train_df.truncate(after=000)
train_df.head()

,file_name,text
0,9195.jpg,නතරවු
1,9196.jpg,විරාමයට
2,9197.jpg,නායයෑම
3,9198.jpg,ප්‍රසන්නභාවය
4,9199.jpg,ආරුමුගන්


In [10]:
test_df = import_dataset('/kaggle/input/real-synth/real_test.tsv')

#train_df = train_df.truncate(after=5000)
test_df.head()

,file_name,text
0,0.jpg,සියල්ලනටම
1,3.jpg,ගැලපෙන්නෙ
2,4.jpg,ෂෝපහාෂය
3,5.jpg,ජේත්තුවට
4,7.jpg,සව්සිරිපායේදී


In [11]:
import torch
from torch.utils.data import Dataset
from PIL import Image

class IAMDataset(Dataset):
    def __init__(self, root_dir, df, processor, max_target_length=128):
        self.root_dir = root_dir
        self.df = df
        self.processor = processor
        self.max_target_length = max_target_length

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        # get file name + text
        file_name = self.df['file_name'][idx]
        if ".jpg" not in file_name:
          file_name = (file_name.split(".")[0]) + ".jpg"
        text = self.df['text'][idx]

        if text != "":
            # prepare image (i.e. resize + normalize)
            image = Image.open(self.root_dir + file_name).convert("RGB")
            pixel_values = self.processor(image, return_tensors="pt").pixel_values
            # add labels (input_ids) by encoding the text
            labels = self.processor.tokenizer(text,
                                              padding="max_length",
                                              max_length=self.max_target_length).input_ids
            # important: make sure that PAD tokens are ignored by the loss function
            labels = [label if label != self.processor.tokenizer.pad_token_id else -100 for label in labels]
    
            encoding = {"pixel_values": pixel_values.squeeze(), "labels": torch.tensor(labels)}
            return encoding

In [12]:
#load dataset processor
from transformers import TrOCRProcessor

processor = TrOCRProcessor.from_pretrained("Ransaka/TrOCR-Sinhala")

In [13]:
train_dataset = IAMDataset(root_dir='/kaggle/input/real-synth/train/',
                           df=train_df,
                           processor=processor)
eval_dataset = IAMDataset(root_dir='/kaggle/input/real-synth/test/',
                           df=test_df,
                           processor=processor)

In [14]:
print(eval_dataset[0])

{'pixel_values': tensor([[[1., 1., 1.,  ..., 1., 1., 1.],
         [1., 1., 1.,  ..., 1., 1., 1.],
         [1., 1., 1.,  ..., 1., 1., 1.],
         ...,
         [1., 1., 1.,  ..., 1., 1., 1.],
         [1., 1., 1.,  ..., 1., 1., 1.],
         [1., 1., 1.,  ..., 1., 1., 1.]],

        [[1., 1., 1.,  ..., 1., 1., 1.],
         [1., 1., 1.,  ..., 1., 1., 1.],
         [1., 1., 1.,  ..., 1., 1., 1.],
         ...,
         [1., 1., 1.,  ..., 1., 1., 1.],
         [1., 1., 1.,  ..., 1., 1., 1.],
         [1., 1., 1.,  ..., 1., 1., 1.]],

        [[1., 1., 1.,  ..., 1., 1., 1.],
         [1., 1., 1.,  ..., 1., 1., 1.],
         [1., 1., 1.,  ..., 1., 1., 1.],
         ...,
         [1., 1., 1.,  ..., 1., 1., 1.],
         [1., 1., 1.,  ..., 1., 1., 1.],
         [1., 1., 1.,  ..., 1., 1., 1.]]]), 'labels': tensor([   2, 6867, 6538, 3715,    3, -100, -100, -100, -100, -100, -100, -100,
        -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100,
        -100, -100, -100, 

In [15]:
print("Number of training examples:", len(train_dataset))
print("Number of validation examples:", len(eval_dataset))

encoding = eval_dataset[0]
for k,v in encoding.items():
  print(k, v.shape)

Number of training examples: 29695
Number of validation examples: 7423
pixel_values torch.Size([3, 384, 384])
labels torch.Size([128])


In [16]:
image = Image.open(train_dataset.root_dir + train_df['file_name'][0]).convert("RGB")
image

labels = encoding['labels']
labels[labels == -100] = processor.tokenizer.pad_token_id
label_str = processor.decode(labels, skip_special_tokens=True)
print(label_str)

සියල්ලනටම


## Model Training

In [17]:
import torch

torch.cuda.empty_cache()
device = torch.device('cuda:0' if torch.cuda.is_available else 'cpu')
model = model.to(device)

In [ ]:
# set special tokens used for creating the decoder_input_ids from the labels
model.config.decoder_start_token_id = processor.tokenizer.cls_token_id
model.config.pad_token_id = processor.tokenizer.pad_token_id
# make sure vocab size is set correctly
model.config.vocab_size = model.config.decoder.vocab_size

# set beam search parameters
model.config.eos_token_id = processor.tokenizer.sep_token_id
model.config.max_length = 64
model.config.early_stopping = True
model.config.no_repeat_ngram_size = 3
model.config.length_penalty = 2.0
model.config.num_beams = 4
model.config.image_size = 128

In [ ]:
class TrainingConfig:
    BATCH_SIZE:    int = 32
    EPOCHS:         int = 1
    LEARNING_RATE: float = 0.00005

In [43]:
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments

In [55]:
training_args = Seq2SeqTrainingArguments(
    predict_with_generate=True,
    eval_strategy='no',
    per_device_train_batch_size=TrainingConfig.BATCH_SIZE,
    per_device_eval_batch_size=TrainingConfig.BATCH_SIZE,
    fp16=True,
    output_dir="./model_training/",
    logging_steps=50,
    save_steps=1000,
    #eval_steps=1,
    report_to='tensorboard',
    num_train_epochs=TrainingConfig.EPOCHS,
    push_to_hub=True,
    hub_model_id = "ds1219/Pandithaya-HW-Words-208-DLoRA",
    hub_private_repo=True,
    resume_from_checkpoint=True,
    #load_best_model_at_end=True,
    metric_for_best_model="cer",
    weight_decay=0.01,
    learning_rate=TrainingConfig.LEARNING_RATE,
)

In [ ]:
from transformers import default_data_collator
import os
from tqdm.notebook import tqdm
import Levenshtein

# Setup for saving checkpoints
checkpoint_dir = "./model_checkpoints/"
os.makedirs(checkpoint_dir, exist_ok=True)

# Define total number of epochs to train
MAX_EPOCHS = 5  # Change this to your desired number of epochs
best_cer = float('inf')
best_epoch = 0

# Function to evaluate model and calculate CER
def evaluate_model(model_to_eval):
    print("Evaluating model...")
    model_to_eval.eval()  # Set model to evaluation mode
    
    pred = []
    truth = []
    
    with torch.no_grad():
        for batch in tqdm(test_dataloader):
            # predict using generate
            pixel_values = batch["pixel_values"].to(device)
            outputs = model_to_eval.generate(pixel_values, interpolate_pos_encoding=True)

            # decode
            pred_str = processor.batch_decode(outputs, skip_special_tokens=True)
            labels = batch["labels"]
            labels[labels == -100] = processor.tokenizer.pad_token_id
            label_str = processor.batch_decode(labels, skip_special_tokens=True)

            pred.extend(pred_str)
            truth.extend(label_str)
    
    # Flatten the lists if they're nested
    flat_pred = [item for sublist in pred for item in (sublist if isinstance(sublist, list) else [sublist])]
    flat_truth = [item for sublist in truth for item in (sublist if isinstance(sublist, list) else [sublist])]
    
    # Calculate CER
    total_edit_distance = 0
    total_characters = 0
    
    for gt, pr in zip(flat_truth, flat_pred):
        total_edit_distance += Levenshtein.distance(gt, pr)
        total_characters += len(gt)
    
    cer = total_edit_distance / total_characters if total_characters > 0 else 0.0
    print(f"Character Error Rate (CER): {cer:.4f}")
    
    return cer

# Create test dataloader for evaluation
test_dataloader = DataLoader(eval_dataset, batch_size=8)

# Custom training loop with CER evaluation after each epoch
for epoch in range(MAX_EPOCHS):
    print(f"\n=== Starting Epoch {epoch+1}/{MAX_EPOCHS} ===")
    
    # Train for one epoch
    training_args.num_train_epochs = 1
    training_args.output_dir = f"{checkpoint_dir}/epoch_{epoch+1}"
    
    # Create a new trainer for this epoch
    trainer = Seq2SeqTrainer(
        model=model,
        tokenizer=processor.image_processor,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        data_collator=default_data_collator
    )
    
    # Train for one epoch
    trainer.train()
    
    # Evaluate with CER
    current_cer = evaluate_model(model)
    
    # Save model if it's the best so far
    if current_cer < best_cer:
        best_cer = current_cer
        best_epoch = epoch + 1
        print(f"New best CER: {best_cer:.4f} at epoch {best_epoch}")
        model_save_path = f"{checkpoint_dir}/best_model"
        model.save_pretrained(model_save_path)
        processor.save_pretrained(model_save_path)
    
    print(f"Current CER: {current_cer:.4f}, Best CER: {best_cer:.4f} (Epoch {best_epoch})")

print(f"\nTraining complete. Best CER: {best_cer:.4f} achieved at epoch {best_epoch}")

# Load the best model after training
if best_epoch > 0:
    print(f"Loading best model from epoch {best_epoch}...")
    model = VisionEncoderDecoderModel.from_pretrained(f"{checkpoint_dir}/best_model")
    model.to(device)
    
    # Final evaluation
    final_cer = evaluate_model(model)
    print(f"Final CER with best model: {final_cer:.4f}")
    
    # Push to Hub
    model.push_to_hub("ds1219/Pandithaya-HW-Words-208-DLoRA-Best")

/tmp/ipykernel_35/3442273077.py:4: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `CustomSeq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = CustomSeq2SeqTrainer(


## Inference

Note that after training, you can easily load the model using the .`from_pretrained(output_dir)` method.

For inference on new images, I refer to my inference notebook, that can also be found in my [Transformers Tutorials repository](https://github.com/NielsRogge/Transformers-Tutorials) on Github.

In [3]:
device = torch.device('cuda:0' if torch.cuda.is_available else 'cpu')

In [6]:
from transformers import TrOCRProcessor, VisionEncoderDecoderModel, AutoTokenizer

# For example, using a TrOCR model pre-trained on handwritten text:
processor = TrOCRProcessor.from_pretrained("Ransaka/TrOCR-Sinhala",
                                           size={"height": 128, "width": 128})
# (Optionally, if you want to change the decoder tokenizer—for example, to a RoBERTa tokenizer for a different language—you can load it separately)
tokenizer = AutoTokenizer.from_pretrained("Ransaka/TrOCR-Sinhala")

# If you want to swap in a different tokenizer for the decoder, assign it to the processor:
processor.tokenizer = tokenizer

# Load the pre-trained TrOCR model:
model = VisionEncoderDecoderModel.from_pretrained("ds1219/Pandithaya-Printed-Words-128")

model.to(device)

preprocessor_config.json:   0%|          | 0.00/325 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/315 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/485k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/988k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.26G [00:00<?, ?B/s]

VisionEncoderDecoderModel(
  (encoder): ViTModel(
    (embeddings): ViTEmbeddings(
      (patch_embeddings): ViTPatchEmbeddings(
        (projection): Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16))
      )
      (dropout): Dropout(p=0.0, inplace=False)
    )
    (encoder): ViTEncoder(
      (layer): ModuleList(
        (0-11): 12 x ViTLayer(
          (attention): ViTAttention(
            (attention): ViTSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=False)
              (key): Linear(in_features=768, out_features=768, bias=False)
              (value): Linear(in_features=768, out_features=768, bias=False)
            )
            (output): ViTSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.0, inplace=False)
            )
          )
          (intermediate): ViTIntermediate(
            (dense): Linear(in_features=768, out_features=3072, bias=True)
            (i

In [54]:
model2.to(device)

VisionEncoderDecoderModel(
  (encoder): ViTModel(
    (embeddings): ViTEmbeddings(
      (patch_embeddings): ViTPatchEmbeddings(
        (projection): Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16))
      )
      (dropout): Dropout(p=0.0, inplace=False)
    )
    (encoder): ViTEncoder(
      (layer): ModuleList(
        (0-11): 12 x ViTLayer(
          (attention): ViTAttention(
            (attention): ViTSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=False)
              (key): Linear(in_features=768, out_features=768, bias=False)
              (value): Linear(in_features=768, out_features=768, bias=False)
            )
            (output): ViTSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.0, inplace=False)
            )
          )
          (intermediate): ViTIntermediate(
            (dense): Linear(in_features=768, out_features=3072, bias=True)
            (i

In [38]:
def read_and_show(image_path):
    """
    :param image_path: String, path to the input image.

    Returns:
        image: PIL Image.
    """
    image = Image.open(image_path).convert('RGB')
    
    return image

In [58]:
def ocr(image):
    # Preprocess image to get pixel values
    pixel_values = processor(image, return_tensors="pt").pixel_values
    # Move input to the same device as the model
    pixel_values = pixel_values.to(model.device)
    generated_ids = model2.generate(pixel_values, interpolate_pos_encoding=True)
    generated_text = processor.batch_decode(generated_ids, skip_special_tokens=True)[0]
    return generated_text

In [76]:
image = read_and_show("/kaggle/input/real-synth/train/10000.jpg")
text = ocr(image)
print(text)

ඝණ ඝණය


In [29]:
trained_model.push_to_hub("ds1219/Pandithaya-TrOCR-Sinhala-Printed")

NameError: name 'trained_model' is not defined

# Evaluate

In [53]:
import pandas as pd

def import_dataset(path):
  df = pd.read_csv(path, sep='\t', usecols=[0,1])
  #df = pd.read_csv(path, sep='\t', usecols=[0,1])
  df.rename(columns={'file': 'file_name', 'labels': 'text'}, inplace=True)
  #df = df.iloc[1:]
  return(df)
    
test_df = import_dataset('/kaggle/input/real-synth/real_test.tsv')
test_df.head()

,file_name,text
0,0.jpg,සියල්ලනටම
1,3.jpg,ගැලපෙන්නෙ
2,4.jpg,ෂෝපහාෂය
3,5.jpg,ජේත්තුවට
4,7.jpg,සව්සිරිපායේදී


In [7]:
from transformers import TrOCRProcessor, VisionEncoderDecoderModel, AutoTokenizer
device = torch.device('cuda:0' if torch.cuda.is_available else 'cpu')

#For example, using a TrOCR model pre-trained on handwritten text:
processor = TrOCRProcessor.from_pretrained("Ransaka/TrOCR-Sinhala",
                                           size={"height": 128, "width": 128})
# (Optionally, if you want to change the decoder tokenizer—for example, to a RoBERTa tokenizer for a different language—you can load it separately)
tokenizer = AutoTokenizer.from_pretrained("Ransaka/TrOCR-Sinhala")

# If you want to swap in a different tokenizer for the decoder, assign it to the processor:
processor.tokenizer = tokenizer

# Load the pre-trained TrOCR model:
model = VisionEncoderDecoderModel.from_pretrained("ds1219/Pandithaya-Handwritten-Words-128-hmfg")

#model = VisionEncoderDecoderModel.from_pretrained("ds1219/Pandithaya-Handwritten-Words-128-hm-augv2-Dlora")
model = model.to(device)

preprocessor_config.json:   0%|          | 0.00/325 [00:00<?, ?B/s]

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


tokenizer_config.json:   0%|          | 0.00/315 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/485k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/988k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.67k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.26G [00:00<?, ?B/s]

Config of the encoder: <class 'transformers.models.vit.modeling_vit.ViTModel'> is overwritten by shared encoder config: ViTConfig {
  "attention_probs_dropout_prob": 0.0,
  "encoder_stride": 16,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.0,
  "hidden_size": 768,
  "image_size": 384,
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "layer_norm_eps": 1e-12,
  "model_type": "vit",
  "num_attention_heads": 12,
  "num_channels": 3,
  "num_hidden_layers": 12,
  "patch_size": 16,
  "pooler_act": "tanh",
  "pooler_output_size": 768,
  "qkv_bias": false,
  "torch_dtype": "float32",
  "transformers_version": "4.51.1"
}

Config of the decoder: <class 'transformers.models.trocr.modeling_trocr.TrOCRForCausalLM'> is overwritten by shared decoder config: TrOCRConfig {
  "activation_dropout": 0.0,
  "activation_function": "gelu",
  "add_cross_attention": true,
  "attention_dropout": 0.0,
  "bos_token_id": 0,
  "classifier_dropout": 0.0,
  "cross_attention_hidden_size": 768,
  "d_mod

In [43]:
model2 = deepcopy(model)

In [44]:
model2.encoder = model2.encoder.merge_and_unload()
model2.decoder = model2.decoder.merge_and_unload()

In [45]:
model2

VisionEncoderDecoderModel(
  (encoder): ViTModel(
    (embeddings): ViTEmbeddings(
      (patch_embeddings): ViTPatchEmbeddings(
        (projection): Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16))
      )
      (dropout): Dropout(p=0.0, inplace=False)
    )
    (encoder): ViTEncoder(
      (layer): ModuleList(
        (0-11): 12 x ViTLayer(
          (attention): ViTAttention(
            (attention): ViTSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=False)
              (key): Linear(in_features=768, out_features=768, bias=False)
              (value): Linear(in_features=768, out_features=768, bias=False)
            )
            (output): ViTSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.0, inplace=False)
            )
          )
          (intermediate): ViTIntermediate(
            (dense): Linear(in_features=768, out_features=3072, bias=True)
            (i

In [46]:
model

VisionEncoderDecoderModel(
  (encoder): PeftModel(
    (base_model): LoraModel(
      (model): ViTModel(
        (embeddings): ViTEmbeddings(
          (patch_embeddings): ViTPatchEmbeddings(
            (projection): Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16))
          )
          (dropout): Dropout(p=0.0, inplace=False)
        )
        (encoder): ViTEncoder(
          (layer): ModuleList(
            (0-11): 12 x ViTLayer(
              (attention): ViTAttention(
                (attention): ViTSelfAttention(
                  (query): lora.Linear(
                    (base_layer): Linear(in_features=768, out_features=768, bias=False)
                    (lora_dropout): ModuleDict(
                      (default): Dropout(p=0.05, inplace=False)
                    )
                    (lora_A): ModuleDict(
                      (default): Linear(in_features=768, out_features=8, bias=False)
                    )
                    (lora_B): ModuleDict(
                 

In [47]:
model2 = model2.to(device)

In [48]:
import torch
from torch.utils.data import Dataset
from PIL import Image

class IAMDataset(Dataset):
    def __init__(self, root_dir, df, processor, max_target_length=128):
        self.root_dir = root_dir
        self.df = df
        self.processor = processor
        self.max_target_length = max_target_length

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        # get file name + text
        file_name = self.df['file_name'][idx]
        if ".jpg" not in file_name:
          file_name = (file_name.split(".")[0]) + ".jpg"
        text = self.df['text'][idx]

        if text != "":
            # prepare image (i.e. resize + normalize)
            image = Image.open(self.root_dir + file_name).convert("RGB")
            pixel_values = self.processor(image, return_tensors="pt").pixel_values
            # add labels (input_ids) by encoding the text
            labels = self.processor.tokenizer(text,
                                              padding="max_length",
                                              max_length=self.max_target_length).input_ids
            # important: make sure that PAD tokens are ignored by the loss function
            labels = [label if label != self.processor.tokenizer.pad_token_id else -100 for label in labels]
    
            encoding = {"pixel_values": pixel_values.squeeze(), "labels": torch.tensor(labels)}
            return encoding

In [49]:
test_dataset = IAMDataset(root_dir='/kaggle/input/real-synth/test/',
                           df=test_df,
                           processor=processor)

from torch.utils.data import DataLoader

test_dataloader = DataLoader(test_dataset, batch_size=8)
batch = next(iter(test_dataloader))
for k,v in batch.items():
  print(k, v.shape)

labels = batch["labels"]
labels[labels == -100] = processor.tokenizer.pad_token_id
label_str = processor.batch_decode(labels, skip_special_tokens=True)
label_str

pixel_values torch.Size([8, 3, 128, 128])
labels torch.Size([8, 128])


['සියල්ලනටම',
 'ගැලපෙන්නෙ',
 'ෂෝපහාෂය',
 'ජේත්තුවට',
 'සව්සිරිපායේදී',
 'සුමනයො',
 'ත්රියාණති',
 'දේසානුරාගය']

In [51]:
from tqdm.notebook import tqdm
print("Running evaluation...")

pred = []
truth = []

def calculate_cer(ground_truth: list, predictions: list) -> float:
    """
    Calculate the Character Error Rate (CER) between two lists of words.
    CER = (Total character edit distance) / (Total number of characters in ground truth)
    
    :param ground_truth: List of correct words
    :param predictions: List of predicted words
    :return: Character Error Rate (CER)
    """
    if len(ground_truth) != len(predictions):
        raise ValueError("Ground truth and predictions lists must have the same length.")
    
    total_edit_distance = 0
    total_characters = 0
    
    for gt, pred in tqdm(zip(ground_truth, predictions)):
        total_edit_distance += Levenshtein.distance(gt, pred)
        total_characters += len(gt)
    
    return total_edit_distance / total_characters if total_characters > 0 else 0.0
    
for batch in tqdm(test_dataloader):
    # predict using generate
    pixel_values = batch["pixel_values"].to(device)
    outputs = model2.generate(pixel_values, interpolate_pos_encoding=True)

    # decode
    pred_str = processor.batch_decode(outputs, skip_special_tokens=True)
    labels = batch["labels"]
    labels[labels == -100] = processor.tokenizer.pad_token_id
    label_str = processor.batch_decode(labels, skip_special_tokens=True)

    pred.append(pred_str)
    truth.append(label_str)

Running evaluation...


  0%|          | 0/928 [00:00<?, ?it/s]

In [52]:
import Levenshtein

cer = calculate_cer(truth, pred)
print(f"Character Error Rate (CER): {cer:.4f}")

0it [00:00, ?it/s]

Character Error Rate (CER): 0.9941


In [44]:
with open("/kaggle/working/example.txt", "w") as file:
    for i in range(0,len(truth)):
        file.write(f"{truth[i]} - {pred[i]}/n")

# merge lora model

# Pruning

In [12]:
!pip install "optimum-intel[neural-compressor]"@git+https://github.com/huggingface/optimum-intel.git

  Cloning https://github.com/huggingface/optimum-intel.git to /tmp/pip-install-pin83w3q/optimum-intel_cca25bae03644614b2e0129383cd68c6
  Running command git clone --filter=blob:none --quiet https://github.com/huggingface/optimum-intel.git /tmp/pip-install-pin83w3q/optimum-intel_cca25bae03644614b2e0129383cd68c6
  Resolved https://github.com/huggingface/optimum-intel.git to commit f2f51c55037497d2bca8ee76ecca0493cae92c97
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.4/44.4 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 424.6/424.6 kB 14.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 57.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 113.6 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 82.7 MB/s eta 0:00:00:00:01
  Created wheel for optimum-intel

In [13]:
from transformers import VisionEncoderDecoderModel, TrOCRProcessor

model_name = "ds1219/Pandithaya-Printed-Words-128-hm"
processor = TrOCRProcessor.from_pretrained("Ransaka/TrOCR-Sinhala",
                                           size={"height": 128, "width": 128})
# (Optionally, if you want to change the decoder tokenizer—for example, to a RoBERTa tokenizer for a different language—you can load it separately)
tokenizer = AutoTokenizer.from_pretrained("Ransaka/TrOCR-Sinhala")

In [14]:
import torch
from torch.utils.data import Dataset
from PIL import Image

class IAMDataset(Dataset):
    def __init__(self, root_dir, df, processor, max_target_length=128):
        self.root_dir = root_dir
        self.df = df
        self.processor = processor
        self.max_target_length = max_target_length

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        # get file name + text
        file_name = self.df['file_name'][idx]
        if ".jpg" not in file_name:
          file_name = (file_name.split(".")[0]) + ".jpg"
        text = self.df['text'][idx]

        if text != "":
            # prepare image (i.e. resize + normalize)
            image = Image.open(self.root_dir + file_name).convert("RGB")
            pixel_values = self.processor(image, return_tensors="pt").pixel_values
            # add labels (input_ids) by encoding the text
            labels = self.processor.tokenizer(text,
                                              padding="max_length",
                                              max_length=self.max_target_length).input_ids
            # important: make sure that PAD tokens are ignored by the loss function
            labels = [label if label != self.processor.tokenizer.pad_token_id else -100 for label in labels]
    
            encoding = {"pixel_values": pixel_values.squeeze(), "labels": torch.tensor(labels)}
            return encoding

In [15]:
import pandas as pd

def import_dataset(path):
  df = pd.read_csv(path, sep='\t', usecols=[0,1])
  return(df)
    
test_df = import_dataset('/kaggle/input/synthetic-word-small/test.tsv')
test_df.head()

,file_name,text
0,0_0.jpg,ද්‍රවශීළ
1,2_0.jpg,බිස්කටි
2,1_0.jpg,පැල්මඩුල්ලටත්
3,3_0.jpg,ගුණාගුණ
4,0_1.jpg,ආදාරකය


In [16]:
test_dataset = IAMDataset(root_dir='/kaggle/input/synthetic-word-small/archive (1)/test/',
                           df=test_df,
                           processor=processor)

from torch.utils.data import DataLoader

test_dataloader = DataLoader(test_dataset, batch_size=8)
batch = next(iter(test_dataloader))
for k,v in batch.items():
  print(k, v.shape)

labels = batch["labels"]
labels[labels == -100] = processor.tokenizer.pad_token_id
label_str = processor.batch_decode(labels, skip_special_tokens=True)
label_str

pixel_values torch.Size([8, 3, 128, 128])
labels torch.Size([8, 128])


['ද්රවශීළ',
 'බිස්කටි',
 'පැල්මඩුල්ලටත්',
 'ගුණාගුණ',
 'ආදාරකය',
 'සැනින්මත්',
 'ස්වාභාවයෙන්ම',
 'සුවිශේසත්වයක්']

In [17]:
from transformers import VisionEncoderDecoderModel, TrOCRProcessor
from optimum.intel.neural_compressor import IncQuantizer, IncQuantizationConfig

quantization_config = IncQuantizationConfig(approach="static")

decoder = model.decoder

# Initialize the quantizer
quantizer = IncQuantizer(model=decoder, config=quantization_config)

# Apply quantization (requires a calibration dataset)
quantizer.quantize(calibration_dataset=test_dataset)

# Save the quantized model
quantizer.save_pretrained("quantized_trocr_model")

ImportError: cannot import name 'get_file_from_repo' from 'transformers.utils' (/usr/local/lib/python3.11/dist-packages/transformers/utils/__init__.py)